In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install backports.zoneinfo

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install tzdata

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import requests
import time
import datetime

import pytz
from datetime import datetime, timezone
from backports.zoneinfo import ZoneInfo



import psycopg2
import psycopg2.extras
from psycopg2.extensions import AsIs

from datetime import datetime, timezone

import dotenv
from dotenv import load_dotenv
import os

import warnings
warnings.filterwarnings("ignore")

In [5]:
def apicalls_get_data():
    global data
    url = 'https://apicalls.io/api/v2/markets/market-info'
    headers = {
    'Authorization': 'Bearer 539|5d9M5TONvuHKNOVYKwrWKT88fsivCirNPSc9nXXf'
    }

    response = requests.request('GET', url, headers=headers)
    data = response.json()
    print(data)

    dict = data['body']
    dataframe = pd.DataFrame(list(dict.items()))
    dataframe =  dataframe.transpose()

    return dataframe

    

In [6]:
dataframe = apicalls_get_data()

{'meta': {'version': 'v1.0', 'status': 200, 'copywrite': 'https://apicalls.io'}, 'body': {'country': 'U.S.', 'marketIndicator': 'Market Closed', 'uiMarketIndicator': 'Market Closed', 'marketCountDown': 'Market Opens in 7H 57M', 'preMarketOpeningTime': 'Jun 28, 2024 04:00 AM ET', 'preMarketClosingTime': 'Jun 28, 2024 09:30 AM ET', 'marketOpeningTime': 'Jun 28, 2024 09:30 AM ET', 'marketClosingTime': 'Jun 28, 2024 04:00 PM ET', 'afterHoursMarketOpeningTime': 'Jun 28, 2024 04:00 PM ET', 'afterHoursMarketClosingTime': 'Jun 28, 2024 08:00 PM ET', 'previousTradeDate': 'Jun 27, 2024', 'nextTradeDate': 'Jul 1, 2024', 'isBusinessDay': True, 'mrktStatus': 'Closed', 'mrktCountDown': 'Opens in 7H 57M'}}


In [60]:
cleaned_df = dataframe.copy()

In [61]:
cleaned_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,country,marketIndicator,uiMarketIndicator,marketCountDown,preMarketOpeningTime,preMarketClosingTime,marketOpeningTime,marketClosingTime,afterHoursMarketOpeningTime,afterHoursMarketClosingTime,previousTradeDate,nextTradeDate,isBusinessDay,mrktStatus,mrktCountDown
1,U.S.,Market Closed,Market Closed,Market Opens in 7H 57M,"Jun 28, 2024 04:00 AM ET","Jun 28, 2024 09:30 AM ET","Jun 28, 2024 09:30 AM ET","Jun 28, 2024 04:00 PM ET","Jun 28, 2024 04:00 PM ET","Jun 28, 2024 08:00 PM ET","Jun 27, 2024","Jul 1, 2024",True,Closed,Opens in 7H 57M


In [62]:
def apicalls_correct_date_time(cleaned_df):

    # Get region data from table
    cwd = os.getcwd()
    file_path = os.path.join(cwd, "Credentials.env")
    load_dotenv(file_path)
    hostname, port_id, database, username, password = (
        os.getenv(key) for key in ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
    )
    print(hostname, port_id, database, username, password)

    cleaned_df. columns=cleaned_df. iloc[0]
    cleaned_df = (cleaned_df.drop(0))
    cleaned_df.reset_index()
    cleaned_df['country'][1] = 'US'


    time_cols = list(cleaned_df.columns[cleaned_df.columns.str.contains(pat = 'Time')])
    date_cols = (list([cleaned_df.columns[cleaned_df.columns.str.contains(pat = 'Date')]]))

    for col in time_cols:
        cleaned_df[col] = pd.to_datetime(cleaned_df[col])#, format='%Y-%m-%d %H:%M:%S') #line 17
    for col in date_cols:
        cleaned_df[col] = cleaned_df[col] + ' 00:00 AM ET'
        print(cleaned_df[col])
        cleaned_df[col] = cleaned_df[col].apply(pd.to_datetime)#, format="%b %d, %Y", errors="coerce").dt.strftime("%d.%m.%Y")

    cleaned_df.assign(financial_market_name="")
    cleaned_df.assign(financial_market_PK="")
    print("Printing cleaned df", cleaned_df)
    market_df = pd.DataFrame()
    for row in cleaned_df.index:
        if row == 0:
            continue
        
        print("now to convert the timezone")
        # Time correction to UTC
        for col in time_cols:
            # print("LocaL OPEN time:", cleaned_df['local_open'][i], "Market name:", cleaned_df['primary_exchanges'][i])
            local  = str(cleaned_df[col][row])
            print ("Got local time: ", local)
            # Get timezone we're trying to convert from
            local_tz = ZoneInfo("America/New_York")
            # UTC timezone
            utc_tz = ZoneInfo("UTC")

            dt = datetime.strptime(local,"%Y-%m-%d %H:%M:%S")
            dt = dt.replace(tzinfo=local_tz)
            dt_open_utc = dt.astimezone(utc_tz)
            dt_open_utc = pd.Timestamp(dt_open_utc)
            cleaned_df[col][row] = dt_open_utc.tz_convert('utc')
            print("Converted to UTC: ", cleaned_df[col][row])
        
        # try: 
        with psycopg2.connect(host=hostname,dbname=database,user=username,password=password,port=port_id) as conn:
            with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
                    query=f'''SELECT "PK"
                    from "dyGEO".country_list_view 
                    where alpha_2_code='{cleaned_df['country'][row]}'; ''' # The country name should be US and not U.S.
                    print(query)
                    cur.execute(query)
                    result = cur.fetchall()
                    print("counrty_PK = ", result)
        # except:
        #     raise Exception('Could not fetch the country PK.')
        # finally:
        if conn is not None:
            conn.close()

        # try: 
        with psycopg2.connect(host=hostname,dbname=database,user=username,password=password,port=port_id) as conn:
            with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
                for pk in result:
                    query=f'''SELECT "timezone_region_city_PK"
                    from "dyGEO".country_timezone_region_city_rel_view 
                    where "country_PK" = {pk[0]}; ''' # The counrty name should be US and not U.S.
                    print(query)
                    cur.execute(query)
                    timezone_region_city_PK_list = cur.fetchall()
                    print("timezone_region_city_PK_list: ", timezone_region_city_PK_list)
        # except:
        #     raise Exception('Could not fetch the timezone region city PK.')
        # finally:
        if conn is not None:
            conn.close()  
        # try: 
        with psycopg2.connect(host=hostname,dbname=database,user=username,password=password,port=port_id) as conn:
            with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
                for tz_PK in range(len(timezone_region_city_PK_list)):
                    query=f''' SELECT "financial_market_PK", financial_market_name
                    from "dyLEARN".financial_market_list 
                    where "financial_market_timezone_region_city_PK" ='{timezone_region_city_PK_list[tz_PK][0]}'; ''' # The counrty name should be US and not U.S.
                    print(query)
                    cur.execute(query)
                    markets = cur.fetchall()
                    num = row
                    for market in markets:
                        financial_market_PK = market[0]
                        financial_market_name = market[1]
                        data_dict = {'financial_market_name': financial_market_name,'financial_market_PK': financial_market_PK,'country':cleaned_df['country'][row], 'marketIndicator':cleaned_df['marketIndicator'][row],
                                    'uiMarketIndicator':cleaned_df['uiMarketIndicator'][row],'marketCountDown':cleaned_df['marketCountDown'][row],
                                    'preMarketOpeningTime':cleaned_df['preMarketOpeningTime'][row], 'preMarketClosingTime':cleaned_df['preMarketClosingTime'][row],
                                    'marketOpeningTime':cleaned_df['marketOpeningTime'][row], 'marketClosingTime':cleaned_df['marketClosingTime'][row],
                                    'afterHoursMarketOpeningTime':cleaned_df['afterHoursMarketOpeningTime'][row], 'afterHoursMarketClosingTime':cleaned_df['afterHoursMarketClosingTime'][row],
                                    'previousTradeDate':cleaned_df['previousTradeDate'][row], 'nextTradeDate':cleaned_df['nextTradeDate'][row],
                                    'isBusinessDay':cleaned_df['isBusinessDay'][row], 'mrktStatus':cleaned_df['mrktStatus'][row], 'mrktCountDown':cleaned_df['mrktCountDown'][row] }
                        temp_df = pd.DataFrame(data_dict, index=[0])
                        # num += 1
                        market_df = pd.concat([market_df,temp_df], ignore_index=True)
        # except:
        #     raise Exception('Could not fetch the market name')
        # finally:
        if conn is not None:
            conn.close()
        
    # cleaned_df = pd.concat([cleaned_df, market_df])#, ignore_index=True)
    # print("Concatinated dataframe:", cleaned_df)

    print(market_df)
    return (market_df)

In [63]:
apicalls_time_adjusted_dataframe = apicalls_correct_date_time(cleaned_df)

database-1.ctzm0hf7fhri.eu-central-1.rds.amazonaws.com 5432 dyDATA_new postgres Proc2023awsrdspostgresql
0         previousTradeDate            nextTradeDate
1  Jun 27, 2024 00:00 AM ET  Jul 1, 2024 00:00 AM ET
Printing cleaned df 0 country marketIndicator uiMarketIndicator         marketCountDown  \
1      US   Market Closed     Market Closed  Market Opens in 7H 57M   

0 preMarketOpeningTime preMarketClosingTime   marketOpeningTime  \
1  2024-06-28 04:00:00  2024-06-28 09:30:00 2024-06-28 09:30:00   

0   marketClosingTime afterHoursMarketOpeningTime afterHoursMarketClosingTime  \
1 2024-06-28 16:00:00         2024-06-28 16:00:00         2024-06-28 20:00:00   

0 previousTradeDate nextTradeDate isBusinessDay mrktStatus    mrktCountDown  
1        2024-06-27    2024-07-01          True     Closed  Opens in 7H 57M  
now to convert the timezone
Got local time:  2024-06-28 04:00:00
Converted to UTC:  2024-06-28 08:00:00+00:00
Got local time:  2024-06-28 09:30:00
Converted to UTC:  2024-0

In [64]:
apicalls_time_adjusted_dataframe

,financial_market_name,financial_market_PK,country,marketIndicator,uiMarketIndicator,marketCountDown,preMarketOpeningTime,preMarketClosingTime,marketOpeningTime,marketClosingTime,afterHoursMarketOpeningTime,afterHoursMarketClosingTime,previousTradeDate,nextTradeDate,isBusinessDay,mrktStatus,mrktCountDown
0,NYSE Arca,39,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M
1,Chicago Board of Trade,5,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M
2,American Stock Exchange,40,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M
3,New York Mercantile Exchange,7,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M
4,NASDAQ,16,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M
5,New York Stock Exchange,20,US,Market Closed,Market Closed,Market Opens in 7H 57M,2024-06-28 08:00:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 13:30:00+00:00,2024-06-28 20:00:00+00:00,2024-06-28 20:00:00+00:00,2024-06-29 00:00:00+00:00,2024-06-27,2024-07-01,True,Closed,Opens in 7H 57M


In [66]:
from datetime import date
this_day = str(date.today())

cwd = os.getcwd()
file_path = os.path.join(cwd, "Credentials.env")
load_dotenv(file_path)
hostname, port_id, database, username, password = (
    os.getenv(key) for key in ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
)
print(hostname, port_id, database, username, password)

# "financial_market_opening_status_UUID"

table_name = '''"dyTRADE".financial_market_session_time_log'''

#Commenting for time till market_PK not ready apicalls_time_adjusted_dataframe['financial_market_PK'][i],

try:
    with psycopg2.connect(host=hostname,dbname=database,user=username,password=password,port=port_id) as conn:
        with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
            for i in range(len(apicalls_time_adjusted_dataframe)):
                if apicalls_time_adjusted_dataframe['mrktStatus'][i] == "Open":
                    apicalls_time_adjusted_dataframe['mrktStatus'][i]=1
                else:
                    apicalls_time_adjusted_dataframe['mrktStatus'][i]=0
                # if apicalls_time_adjusted_dataframe['notes'][i] is None:
                #     apicalls_time_adjusted_dataframe['notes'][i] = " "
                ################################################
                id= '''"financial_market_session_time_ID"'''
                financial_market_PK= '''"financial_market_session_time_financial_market_PK"'''
                session_time_log_utc_date= '''"financial_market_session_time_log_utc_date"'''
                opening_UTC_time= '''"financial_market_session_time_opening_UTC_time"'''
                closure_UTC_time= '''"financial_market_session_time_closure_UTC_time"'''
                # session_time_note= '''"financial_market_session_time_note"'''
                market_status= '''"financial_market_session_time_activity_status"'''
                market_session_PK =  '''"financial_market_session_time_market_session_PK"'''
                next_trade_session_date = '''"financial_market_session_next_market_trading_session_date"'''
                
                query=f'''INSERT INTO {table_name} ({id},{financial_market_PK},{market_session_PK},{session_time_log_utc_date},{opening_UTC_time},{closure_UTC_time},{market_status}, {next_trade_session_date})
                VALUES ({apicalls_time_adjusted_dataframe['financial_market_PK'][i]},{apicalls_time_adjusted_dataframe['financial_market_PK'][i]},2,'{this_day}','{apicalls_time_adjusted_dataframe['marketOpeningTime'][i]}','{apicalls_time_adjusted_dataframe['marketClosingTime'][i]}'
                ,{apicalls_time_adjusted_dataframe['mrktStatus'][i]}, '{apicalls_time_adjusted_dataframe['nextTradeDate'][i]}')'''
                print(query)
                cur.execute(query)
            
                # Adding the pre Market data
                query=f'''INSERT INTO {table_name} ({id},{financial_market_PK},{market_session_PK},{session_time_log_utc_date},{opening_UTC_time},{closure_UTC_time},{market_status}, {next_trade_session_date})
                VALUES ({apicalls_time_adjusted_dataframe['financial_market_PK'][i]},{apicalls_time_adjusted_dataframe['financial_market_PK'][i]},1,'{this_day}','{apicalls_time_adjusted_dataframe['preMarketOpeningTime'][i]}','{apicalls_time_adjusted_dataframe['preMarketClosingTime'][i]}'
                ,{apicalls_time_adjusted_dataframe['mrktStatus'][i]}, '{apicalls_time_adjusted_dataframe['nextTradeDate'][i]}')'''
                print(query)
                cur.execute(query)

                # Adding the values of after Market data
                query=f'''INSERT INTO {table_name} ({id},{financial_market_PK},{market_session_PK},{session_time_log_utc_date},{opening_UTC_time},{closure_UTC_time},{market_status}, {next_trade_session_date})
                VALUES ({apicalls_time_adjusted_dataframe['financial_market_PK'][i]},{apicalls_time_adjusted_dataframe['financial_market_PK'][i]},3,'{this_day}','{apicalls_time_adjusted_dataframe['afterHoursMarketOpeningTime'][i]}','{apicalls_time_adjusted_dataframe['afterHoursMarketClosingTime'][i]}'
                ,{apicalls_time_adjusted_dataframe['mrktStatus'][i]}, '{apicalls_time_adjusted_dataframe['nextTradeDate'][i]}')'''
                print(query)
                cur.execute(query)
except:
    raise Exception("The query was not able to write into the financial")

database-1.ctzm0hf7fhri.eu-central-1.rds.amazonaws.com 5432 dyDATA_new postgres Proc2023awsrdspostgresql
INSERT INTO "dyTRADE".financial_market_session_time_log ("financial_market_session_time_ID","financial_market_session_time_financial_market_PK","financial_market_session_time_market_session_PK","financial_market_session_time_log_utc_date","financial_market_session_time_opening_UTC_time","financial_market_session_time_closure_UTC_time","financial_market_session_time_activity_status", "financial_market_session_next_market_trading_session_date")
                VALUES (39,39,2,'2024-06-28','2024-06-28 13:30:00+00:00','2024-06-28 20:00:00+00:00'
                ,0, '2024-07-01 00:00:00')
INSERT INTO "dyTRADE".financial_market_session_time_log ("financial_market_session_time_ID","financial_market_session_time_financial_market_PK","financial_market_session_time_market_session_PK","financial_market_session_time_log_utc_date","financial_market_session_time_opening_UTC_time","financial_marke